# MoEInfra: GPU Memory Profiling on Tesla T4 (16GB)

**Target Environment:** Kaggle 1x Tesla T4 (16GB VRAM, ~15.1 GiB usable), PCIe Gen3 x16 (PHB, no NVLink)  
**Target Model:** INT4 Mixtral-8x7B-v0.1 (~24GB total model, 32 layers, 8 experts/layer, top-2 routing)  
**Scope:** Progressively profile the ACTUAL current `InferenceEngine` and subsystems (`CacheManager`, `TransferScheduler`, `PrefetchEngine`, `ExpertRouter`, `ModelLoader`) as implemented in the repository, using the real `InferenceEngine.generate()` and `ForwardRequest` APIs.

---

### Profiling Progression
1. **Hardware & Environment Inspection:** Detect GPU capabilities, driver status, and initial `nvidia-smi` state.
2. **Clean CUDA Memory Baseline:** Establish a clean reference baseline for VRAM tracking.
3. **MoEInfra Configuration & Engine Initialization:** Load `config.yaml` and initialize the actual `InferenceEngine` and subsystems.
4. **Mixtral INT4 Memory Budget & Subsystem Allocation:** Analyze the exact INT4 Mixtral-8x7B parameters and measure static memory.
5. **Expert Cache Residency Inspection:** Query `CacheManager.stats()` and inspect GPU vs CPU tier residency.
6. **Reusable CUDA Memory Checkpoint Helper:** Implement `CUDAMemoryTracker` for high-precision stage profiling.
7. **Tokenizer & Proper ForwardRequest API:** Load tokenizer and prepare real `ForwardRequest` inputs.
8. **Inference Execution & Memory Lifecycle:** Execute `InferenceEngine.generate()` with memory checkpoints (stopping at the current `_prefill` stub).
9. **Larger Context Request Staging:** Profile memory during larger prompt staging and generation invocation.
10. **Low-Level Allocator Breakdown (`torch.cuda.memory_summary()`):** Inspect active blocks, reserved segments, and fragmentation.
11. **CUDA Memory History Snapshot Export:** Capture PyTorch 2.1+ memory history and export `cuda_memory_snapshot.pickle`.
12. **PyTorch Profiler (`torch.profiler`):** Capture CUDA and CPU activity, exporting `moeinfra_inference_trace.json`.
13. **Milestone Summary Table & T4 Headroom Analysis:** Consolidate all measurements and evaluate headroom against the 16GB T4 limit.
14. **NVIDIA Nsight Systems Preparation Guide:** Setup commands and NVTX instrumentation for full PCIe-to-Compute tracing.


## 1. Environment & GPU Information (`nvidia-smi`)

Inspect the host hardware, PyTorch build, CUDA runtime, and initial GPU memory status.


In [ ]:
import os
import sys
import gc
import time
import subprocess
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_id = 0
    props = torch.cuda.get_device_properties(device_id)
    total_mem_gb = props.total_memory / (1024 ** 3)
    free_mem_gb, _ = [x / (1024 ** 3) for x in torch.cuda.mem_get_info(device_id)]
    print(f"Device [{device_id}]    : {props.name}")
    print(f"Compute Cap    : {props.major}.{props.minor}")
    print(f"MultiProcessors: {props.multi_processor_count}")
    print(f"Total VRAM     : {total_mem_gb:.2f} GB ({props.total_memory / (1024**2):,.0f} MB)")
    print(f"Initial Free   : {free_mem_gb:.2f} GB")
else:
    print("WARNING: CUDA is not available. Profiling will report CPU fallback metrics.")

# Execute nvidia-smi
print("\n--- nvidia-smi Output ---")
try:
    smi_output = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.used,memory.free,utilization.gpu", "--format=csv"],
        stderr=subprocess.STDOUT
    ).decode("utf-8")
    print(smi_output)
except Exception as e:
    print(f"Note: nvidia-smi not accessible directly via subprocess ({e}).")
    try:
        get_ipython().system('nvidia-smi')
    except Exception:
        print("Running in non-interactive environment without nvidia-smi command.")


## 2. Clean CUDA Memory Baseline

Establish a pristine reference baseline by forcing garbage collection, clearing PyTorch's caching allocator, and resetting peak memory counters.


In [ ]:
def reset_cuda_memory(device: int = 0):
    """Force garbage collection, release cached blocks, and reset peak statistics."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)

reset_cuda_memory(0)

if torch.cuda.is_available():
    base_allocated_mb = torch.cuda.memory_allocated(0) / (1024 ** 2)
    base_reserved_mb = torch.cuda.memory_reserved(0) / (1024 ** 2)
    base_max_alloc_mb = torch.cuda.max_memory_allocated(0) / (1024 ** 2)
    print(f"[Baseline] Memory Allocated: {base_allocated_mb:.2f} MB")
    print(f"[Baseline] Memory Reserved : {base_reserved_mb:.2f} MB (cached by PyTorch allocator)")
    print(f"[Baseline] Peak Allocated   : {base_max_alloc_mb:.2f} MB")
else:
    print("[Baseline] CUDA not available; baseline memory is 0 MB.")


## 3. Load MoEInfra Configuration and InferenceEngine

Import and instantiate the actual project components:
- `config_loader.load_config("config.yaml")`
- `engine.engine.InferenceEngine`
- `cache.manager.CacheManager`
- `transfer.scheduler.TransferScheduler`
- `engine.prefetch.PrefetchEngine`
- `model.loader.ModelLoader`


In [ ]:
from config_loader import load_config, get_section
from engine.engine import InferenceEngine
from engine.types import ForwardRequest, ForwardResult, PrefetchPolicy
from cache.manager import CacheManager
from cache.types import EvictionPolicy
from transfer.scheduler import TransferScheduler
from engine.prefetch import PrefetchEngine
from model.loader import ModelLoader

# Locate and load config.yaml
config_path = PROJECT_ROOT / "config.yaml"
if not config_path.is_file():
    config_path = Path("config.yaml").resolve()

print(f"Loading configuration from: {config_path}")
cfg = load_config(str(config_path))

print("\n--- Configuration Summary ---")
print(f"Model       : {cfg['model']['name']}")
print(f"Architecture: {cfg['model']['num_layers']} layers, {cfg['model']['num_experts']} experts/layer, top-{cfg['model']['num_experts_per_tok']} routing")
print(f"Dimensions  : hidden_size={cfg['model']['hidden_size']}, intermediate_size={cfg['model']['intermediate_size']}, quant={cfg['model']['quantization']}")
print(f"Cache Slots : GPU={cfg['cache']['gpu_slots']} slots, CPU={cfg['cache']['cpu_slots']} slots, policy={cfg['cache']['eviction_policy']}")
print(f"PCIe Limits : {cfg['transfer']['bandwidth_gbps']} GB/s peak, prefetch_depth={cfg['transfer']['prefetch_depth']}")

# Initialize top-level InferenceEngine exactly as implemented in the repository
engine = InferenceEngine(cfg)
print(f"\nInitialized Engine: {engine}")

# Wire up the actual subsystems with config parameters
cache_policy = EvictionPolicy(cfg["cache"]["eviction_policy"])
cache_mgr = CacheManager(
    gpu_slots=cfg["cache"]["gpu_slots"],
    cpu_slots=cfg["cache"]["cpu_slots"],
    policy=cache_policy,
)

transfer_sched = TransferScheduler(
    cache_manager=cache_mgr,
    bandwidth_gbps=cfg["transfer"]["bandwidth_gbps"],
    max_concurrent=cfg["transfer"]["max_concurrent_transfers"],
)

prefetch_policy = PrefetchPolicy.LOOKAHEAD if cfg["transfer"]["prefetch_depth"] > 1 else PrefetchPolicy.NEXT_LAYER
prefetch_eng = PrefetchEngine(
    cache_manager=cache_mgr,
    transfer_scheduler=transfer_sched,
    policy=prefetch_policy,
    depth=cfg["transfer"]["prefetch_depth"],
)

model_loader = ModelLoader(
    model_name=cfg["model"]["name"],
    num_layers=cfg["model"]["num_layers"],
    num_experts=cfg["model"]["num_experts"],
    hidden_size=cfg["model"]["hidden_size"],
    intermediate_size=cfg["model"]["intermediate_size"],
    quantization=cfg["model"]["quantization"],
)
model_loader.load()

# Assign the real subsystems to the engine instance
engine._cache_manager = cache_mgr
engine._transfer_scheduler = transfer_sched
engine._prefetch_engine = prefetch_eng
engine._model = model_loader

print("Subsystems successfully wired to InferenceEngine.")


## 4. Mixtral INT4 Memory Calculations & Hardware Budget

### Accurate Parameter & Memory Analysis for Mixtral-8x7B:
Mixtral-8x7B has **46.7B total parameters** (~12.9B active per token across 32 layers).

#### Expert SwiGLU Dimensions:
Each expert contains three projection matrices ($w_1, w_2, w_3$):
- $w_1$ (gate): $4096 \times 14336 = 58,720,256$ parameters
- $w_2$ (down): $14336 \times 4096 = 58,720,256$ parameters
- $w_3$ (up)  : $4096 \times 14336 = 58,720,256$ parameters
- **Parameters per expert:** $3 \times 4096 \times 14336 = 176,160,768$ parameters (~176.16M params)
- **Total experts:** $32 \text{ layers} \times 8 \text{ experts/layer} = 256 \text{ experts}$
- **Total expert parameters:** $256 \times 176.16\text{M} = 45.097\text{B parameters}$

#### Memory Footprint Comparison:
| Precision | Bytes/Param | Per-Expert Size | Total 256 Experts | Fits on 16GB T4? |
| :--- | :--- | :--- | :--- | :--- |
| **FP32** | 4.0 bytes | **672.0 MiB** (704.6 MB) | **168.0 GiB** (180.4 GB) | ❌ No |
| **FP16** | 2.0 bytes | **336.0 MiB** (352.3 MB) | **84.0 GiB** (90.2 GB) | ❌ No |
| **INT4** | 0.5 bytes | **84.00 MiB** (88.08 MB) | **21.00 GiB** (22.55 GB) | ❌ No (~24GB total model) |

#### Why Expert Offloading is Mandatory on Single T4 (16GB):
- A single Tesla T4 has 16GB VRAM (~15.1 GiB usable, ~15,360 MB usable space).
- The **~24GB INT4 model exceeds the 16GB limit**.
- **MoEInfra Solution:**
  - **GPU Cache (`gpu_slots = 8`):** $8 \times 84.00\text{ MiB} = \mathbf{672.00\text{ MiB}} \approx 704.64\text{ MB}$.
  - **CPU Cache (`cpu_slots = 32`):** $32 \times 84.00\text{ MiB} = \mathbf{2,688.00\text{ MiB}} \approx 2.625\text{ GiB}$ in pinned host RAM.
  - **Shared Non-Expert Layers (Attention + Embeddings):** $\approx 1.5 - 2.5\text{ GB}$ on GPU.
  - **Total Static GPU Footprint:** $\approx 2.2 - 3.2\text{ GB}$.
  - **Available T4 Headroom:** Over **12 - 13 GB** remaining for KV cache, batching, activations, and PyTorch workspace!


In [ ]:
# Verify ModelLoader expert size computation
expert_bytes = model_loader.get_expert_size_bytes()
expert_mib = expert_bytes / (1024 ** 2)
expert_mb = expert_bytes / 1e6

total_experts = cfg["model"]["num_layers"] * cfg["model"]["num_experts"]
all_experts_gib = (expert_bytes * total_experts) / (1024 ** 3)
gpu_slots_mib = (expert_bytes * cfg["cache"]["gpu_slots"]) / (1024 ** 2)
cpu_slots_gib = (expert_bytes * cfg["cache"]["cpu_slots"]) / (1024 ** 3)

print("--- Exact INT4 Mixtral-8x7B Memory Calculations ---")
print(f"Per-Expert Size          : {expert_bytes:,} bytes ({expert_mib:.2f} MiB / {expert_mb:.2f} MB)")
print(f"All 256 Experts (INT4)   : {all_experts_gib:.2f} GiB")
print(f"GPU Cache ({cfg['cache']['gpu_slots']} slots)       : {gpu_slots_mib:.2f} MiB (~{gpu_slots_mib / 1024:.2f} GiB)")
print(f"CPU Cache ({cfg['cache']['cpu_slots']} slots)      : {cpu_slots_gib:.2f} GiB")

if torch.cuda.is_available():
    torch.cuda.synchronize(0)
    current_alloc_mb = torch.cuda.memory_allocated(0) / (1024 ** 2)
    current_res_mb = torch.cuda.memory_reserved(0) / (1024 ** 2)
    print(f"\nCurrent GPU Memory Allocated: {current_alloc_mb:.2f} MB")
    print(f"Current GPU Memory Reserved : {current_res_mb:.2f} MB")


## 5. Inspect Current GPU/CPU Expert-Cache Residency

Inspect the state of `engine._cache_manager` across both tiers (`_gpu_cache` and `_cpu_cache`).


In [ ]:
import pandas as pd

stats = engine._cache_manager.stats()
print("--- CacheManager Statistics ---")
print(f"GPU Slots Budget : {engine._cache_manager.gpu_slots}")
print(f"CPU Slots Budget : {engine._cache_manager.cpu_slots}")
print(f"GPU Slots Used   : {stats.gpu_slots_used}")
print(f"CPU Slots Used   : {stats.cpu_slots_used}")
print(f"Cache Hits       : {stats.hits}")
print(f"Cache Misses     : {stats.misses}")
print(f"Cache Evictions  : {stats.evictions}")
print(f"Hit Rate         : {stats.hit_rate * 100:.1f}%")

# Inspect residency dictionaries
gpu_items = list(engine._cache_manager._gpu_cache.items())
cpu_items = list(engine._cache_manager._cpu_cache.items())

print(f"\nResident GPU Cache Entries: {len(gpu_items)}")
print(f"Resident CPU Cache Entries: {len(cpu_items)}")

residency_rows = []
for (l, e), entry in gpu_items:
    residency_rows.append({"Tier": "GPU", "Layer": l, "Expert": e, "Device": entry.device, "Size (KB)": entry.size_bytes // 1024, "Accesses": entry.access_count})
for (l, e), entry in cpu_items:
    residency_rows.append({"Tier": "CPU", "Layer": l, "Expert": e, "Device": entry.device, "Size (KB)": entry.size_bytes // 1024, "Accesses": entry.access_count})

df_residency = pd.DataFrame(residency_rows)
if not df_residency.empty:
    display(df_residency)
else:
    print("(Cache currently empty before inference or warm-up)")


## 6. Reusable CUDA Memory Checkpoint Helper (`CUDAMemoryTracker`)

Define a reusable profiling class to record:
- **Allocated Memory**: Exact memory used by live PyTorch tensors.
- **Reserved Memory**: Total memory held by PyTorch's caching allocator from CUDA driver.
- **Peak Allocated & Reserved**: High-water marks reached during forward pass computation.
- **Deltas**: Memory delta compared to baseline and previous stage.


In [ ]:
from contextlib import contextmanager

class CUDAMemoryTracker:
    """Reusable utility for tracking CUDA memory checkpoints across pipeline phases."""

    def __init__(self, device: int = 0):
        self.device = device
        self.checkpoints = []
        self.baseline_allocated = 0.0
        self.baseline_reserved = 0.0
        self.set_baseline("Initial Baseline")

    def set_baseline(self, name: str = "Baseline"):
        if torch.cuda.is_available():
            torch.cuda.synchronize(self.device)
            self.baseline_allocated = torch.cuda.memory_allocated(self.device) / (1024 ** 2)
            self.baseline_reserved = torch.cuda.memory_reserved(self.device) / (1024 ** 2)
        self.checkpoints = []
        self.checkpoint(name)

    def checkpoint(self, stage_name: str, metadata: dict = None) -> dict:
        """Record current and peak memory stats at a named stage."""
        if torch.cuda.is_available():
            torch.cuda.synchronize(self.device)
            alloc_mb = torch.cuda.memory_allocated(self.device) / (1024 ** 2)
            res_mb = torch.cuda.memory_reserved(self.device) / (1024 ** 2)
            max_alloc_mb = torch.cuda.max_memory_allocated(self.device) / (1024 ** 2)
            max_res_mb = torch.cuda.max_memory_reserved(self.device) / (1024 ** 2)
            free_driver_mb, total_driver_mb = [x / (1024 ** 2) for x in torch.cuda.mem_get_info(self.device)]
        else:
            alloc_mb = res_mb = max_alloc_mb = max_res_mb = 0.0
            free_driver_mb = total_driver_mb = 16384.0

        prev_alloc = self.checkpoints[-1]["Allocated (MB)"] if self.checkpoints else alloc_mb
        delta_prev = alloc_mb - prev_alloc
        delta_base = alloc_mb - self.baseline_allocated

        entry = {
            "Stage": stage_name,
            "Allocated (MB)": round(alloc_mb, 2),
            "Reserved (MB)": round(res_mb, 2),
            "Peak Alloc (MB)": round(max_alloc_mb, 2),
            "Peak Res (MB)": round(max_res_mb, 2),
            "Δ vs Prev (MB)": round(delta_prev, 2),
            "Δ vs Base (MB)": round(delta_base, 2),
            "Driver Free (MB)": round(free_driver_mb, 2),
            "Timestamp": round(time.time(), 3),
        }
        if metadata:
            entry.update(metadata)
        self.checkpoints.append(entry)
        return entry

    @contextmanager
    def track(self, stage_name: str):
        """Context manager to measure peak and post memory of a code block."""
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(self.device)
        self.checkpoint(f"{stage_name} [START]")
        yield
        self.checkpoint(f"{stage_name} [END]")

    def to_dataframe(self) -> pd.DataFrame:
        """Return all checkpoints as a pandas DataFrame."""
        return pd.DataFrame(self.checkpoints)

# Instantiate memory tracker
tracker = CUDAMemoryTracker(0)
print(f"Memory tracker initialized with baseline at {tracker.baseline_allocated:.2f} MB allocated.")


## 7. Tokenizer and Proper ForwardRequest API

Prepare real inference inputs using the actual `ForwardRequest` API:
- Prompt text tokenized via `transformers.AutoTokenizer` for `mistralai/Mixtral-8x7B-v0.1` (with fallback for offline/unauthenticated environments).
- Encapsulated in `ForwardRequest(input_ids, attention_mask, max_new_tokens)`.


In [ ]:
# 1. Initialize Tokenizer for Mixtral
model_name = cfg["model"]["name"]
try:
    from transformers import AutoTokenizer
    print(f"Attempting to load tokenizer for {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print("Tokenizer successfully loaded from Hugging Face.")
except Exception as e:
    print(f"Note: Remote tokenizer not loaded ({e}). Using deterministic local fallback.")
    class StandaloneTokenizer:
        """Standard tokenization fallback matching Mixtral vocabulary specs."""
        def __init__(self):
            self.vocab_size = 32000
            self.pad_token_id = 0
            self.eos_token_id = 2
        def encode(self, text: str, return_tensors="pt"):
            words = text.split()
            tokens = [abs(hash(w)) % (self.vocab_size - 10) + 10 for w in words]
            if return_tensors == "pt":
                return torch.tensor([tokens], dtype=torch.long)
            return tokens
        def decode(self, token_ids):
            return " ".join([f"tok_{t}" for t in token_ids])
        def __call__(self, text: str, return_tensors="pt"):
            t = self.encode(text, return_tensors=return_tensors)
            return {"input_ids": t, "attention_mask": torch.ones_like(t)}
    tokenizer = StandaloneTokenizer()

# 2. Prepare Tiny Request
prompt = "Explain Mixture of Experts in one sentence."
encoded = tokenizer(prompt, return_tensors="pt")

tiny_request = ForwardRequest(
    input_ids=encoded["input_ids"],
    attention_mask=encoded.get("attention_mask", None),
    max_new_tokens=4
)

print(f"\nPrompt                : {prompt!r}")
print(f"Input IDs Shape       : {tiny_request.input_ids.shape} ({tiny_request.input_ids.shape[-1]} tokens)")
print(f"Max New Tokens        : {tiny_request.max_new_tokens}")
print(f"ForwardRequest Object : {tiny_request}")


## 8. Inference Execution & Memory Lifecycle

Now we profile the actual `InferenceEngine.generate()` method call as implemented in `engine/engine.py`.

### Architecture Status:
In `engine/engine.py`:
- `generate(request)` starts stage timing, then calls `self._prefill(request.input_ids)` and `self._decode(hidden, ...)`.
- Currently, `_prefill` and `_decode` in `engine/engine.py` are stubs that raise:
  `NotImplementedError: "_prefill: implement expert-offloaded Mixtral prefill"`.
- We profile the exact memory up to this stage and catch the expected `NotImplementedError` rather than fabricating an artificial execution path.


In [ ]:
tracker.set_baseline("Pre-Inference Baseline")

tracker.checkpoint("Staging ForwardRequest", {
    "Batch Size": tiny_request.input_ids.shape[0],
    "Seq Len": tiny_request.input_ids.shape[1]
})

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(0)

# Call the actual InferenceEngine.generate() API
print("Calling engine.generate(tiny_request)...")
inference_error = None
try:
    result = engine.generate(tiny_request)
    print(f"Inference succeeded! Result: {result}")
except NotImplementedError as e:
    inference_error = e
    print(f"\n[Expected Repository State] engine.generate() reached engine._prefill stub:")
    print(f"  -> Caught NotImplementedError: {e}")
    print("  -> Stopping at this last valid profiling stage per repository specification.")
except Exception as e:
    inference_error = e
    print(f"Unexpected error: {e}")

tracker.checkpoint("Post generate() Call Attempt", {"Error Caught": type(inference_error).__name__ if inference_error else "None"})

# Post-attempt cleanup check
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
tracker.checkpoint("Post Cleanup")

df_tiny = tracker.to_dataframe()
print("\n--- Tiny Inference Memory Checkpoints ---")
display(df_tiny[["Stage", "Allocated (MB)", "Reserved (MB)", "Peak Alloc (MB)", "Δ vs Base (MB)"]])


## 9. Larger Context Request Profiling in pandas DataFrame

Profile memory behavior when staging a larger inference workload (e.g. 64 prompt tokens, 16 max new tokens).


In [ ]:
tracker_large = CUDAMemoryTracker(0)
tracker_large.set_baseline("Large Request Baseline")

# Create a realistic 64-token prompt
long_prompt = "Mixture of Experts architecture allows deep neural networks to scale parameter counts while keeping compute per token constant. " * 4
long_encoded = tokenizer(long_prompt, return_tensors="pt")
prompt_len = min(long_encoded["input_ids"].shape[-1], 64)
long_input_ids = long_encoded["input_ids"][:, :prompt_len]

large_request = ForwardRequest(
    input_ids=long_input_ids,
    attention_mask=torch.ones_like(long_input_ids),
    max_new_tokens=16
)

tracker_large.checkpoint("Staged 64-token Request", {
    "Prompt Tokens": prompt_len,
    "Tensor Bytes": large_request.input_ids.nelement() * large_request.input_ids.element_size()
})

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(0)

print(f"Calling engine.generate(large_request) with {prompt_len} tokens...")
try:
    _ = engine.generate(large_request)
except NotImplementedError as e:
    print(f"  -> Caught expected engine stub: {e}")

tracker_large.checkpoint("Post Large generate() Attempt")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
tracker_large.checkpoint("Post Large Cleanup")

df_large = tracker_large.to_dataframe()
print("\n--- Larger Context Request Staging Table ---")
display(df_large[["Stage", "Allocated (MB)", "Reserved (MB)", "Peak Alloc (MB)", "Δ vs Base (MB)"]])


## 10. Low-Level Allocator Breakdown (`torch.cuda.memory_summary()`)

`torch.cuda.memory_summary()` provides comprehensive visibility into PyTorch's caching allocator:
- **Active Memory:** Currently allocated live tensors.
- **Reserved Segments:** OS pages mapped to the CUDA context.
- **Non-releasable Memory:** Memory retained due to internal fragmentation.
- **Allocation Retries:** Indicates whether the allocator struggled and called `cudaFree`/`cudaMalloc` synchronously.


In [ ]:
if torch.cuda.is_available():
    print("=" * 80)
    print("TORCH CUDA MEMORY SUMMARY (Device 0: Tesla T4)")
    print("=" * 80)
    print(torch.cuda.memory_summary(device=0, abbreviated=False))
else:
    print("torch.cuda.memory_summary() requires an active CUDA device.")


### Key Takeaways from Memory Summary:
1. **Reserved vs. Allocated Gap:** The difference between Reserved and Allocated represents the allocator's caching pool. On the T4 (16GB), a moderate pool (~200-500 MB) avoids expensive runtime calls to `cudaMalloc`.
2. **Small vs. Large Pool Allocations:** Intermediate activation tensors (<1MB) land in the small allocation pool, while expert weight tensors (>10MB) use large block segments.
3. **Allocation Retries:** Must remain 0. Retries indicate VRAM exhaustion forcing synchronous cache flushes.


## 11. CUDA Memory History and Visual Snapshot Export

PyTorch 2.1+ supports recording memory allocation history.
We capture a memory snapshot and export it to `benchmarks/cuda_memory_snapshot.pickle`.
This file can be dragged into the [PyTorch Memory Visualizer](https://pytorch.org/memory_viz) to inspect the timeline of allocations.


In [ ]:
snapshot_path = PROJECT_ROOT / "benchmarks" / "cuda_memory_snapshot.pickle"
snapshot_path.parent.mkdir(parents=True, exist_ok=True)

if torch.cuda.is_available():
    try:
        print("Recording CUDA memory history...")
        torch.cuda.memory._record_memory_history(max_entries=100000)

        # Stage request and trigger engine pipeline
        try:
            _ = engine.generate(tiny_request)
        except NotImplementedError:
            pass
        torch.cuda.synchronize(0)

        # Dump snapshot
        torch.cuda.memory._dump_snapshot(str(snapshot_path))
        torch.cuda.memory._record_memory_history(enabled=None)  # Stop recording

        file_size_kb = os.path.getsize(snapshot_path) / 1024
        print(f"Memory snapshot successfully saved: {snapshot_path}")
        print(f"Snapshot file size: {file_size_kb:.1f} KB")
        print("\nTo visualize this snapshot:")
        print("1. Download 'benchmarks/cuda_memory_snapshot.pickle'.")
        print("2. Open your browser to: https://pytorch.org/memory_viz")
        print("3. Drag and drop the file to inspect the allocation timeline.")
    except Exception as e:
        print(f"Memory history recording encountered an error: {e}")
else:
    print("CUDA memory snapshot requires an active CUDA device.")


## 12. PyTorch Profiler (`torch.profiler`) for CUDA & Memory Trace

Use `torch.profiler` to capture microsecond-accurate GPU kernel activity, memory allocations, and tensor shapes. Export the trace to `benchmarks/moeinfra_inference_trace.json`.


In [ ]:
from torch.profiler import profile, record_function, ProfilerActivity

trace_path = PROJECT_ROOT / "benchmarks" / "moeinfra_inference_trace.json"

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

print(f"Profiling with activities: {[a.name for a in activities]}...")

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_stack=True
) as prof:
    with record_function("moeinfra_engine_pipeline"):
        try:
            _ = engine.generate(tiny_request)
        except NotImplementedError:
            pass
    if torch.cuda.is_available():
        torch.cuda.synchronize(0)

# Export Chrome trace format
prof.export_chrome_trace(str(trace_path))
trace_size_kb = os.path.getsize(trace_path) / 1024
print(f"Chrome trace exported to: {trace_path} ({trace_size_kb:.1f} KB)")

# Display Top Operators by Memory
print("\n--- Top Operators by Memory Usage ---")
if torch.cuda.is_available():
    print(prof.key_averages().table(sort_by="cuda_memory_usage", row_limit=8))
else:
    print(prof.key_averages().table(sort_by="cpu_memory_usage", row_limit=8))

# Display Top Operators by Time
print("\n--- Top Operators by Execution Time ---")
sort_metric = "cuda_time_total" if torch.cuda.is_available() else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_metric, row_limit=8))


### How to Inspect the Chrome Trace:
1. Open Google Chrome or Chromium and navigate to `chrome://tracing` (or visit [ui.perfetto.dev](https://ui.perfetto.dev)).
2. Click **Load** and select `benchmarks/moeinfra_inference_trace.json`.
3. Locate the `moeinfra_engine_pipeline` block to observe execution flow and memory allocations.


## 13. Final Milestone Summary Table & T4 Headroom Analysis

Consolidate all memory metrics recorded across the notebook milestones and evaluate the safety margin on a Kaggle 16GB Tesla T4.


In [ ]:
t4_total_capacity_mb = 16.0 * 1024.0  # 16,384 MB (16 GB)

# 1. Measured Milestones
summary_records = [
    {
        "Milestone": "1. Host Driver Clean Baseline",
        "Allocated (MB)": df_tiny.iloc[0]["Allocated (MB)"] if not df_tiny.empty else 0.0,
        "Reserved (MB)": df_tiny.iloc[0]["Reserved (MB)"] if not df_tiny.empty else 0.0,
        "Peak Alloc (MB)": df_tiny.iloc[0]["Peak Alloc (MB)"] if not df_tiny.empty else 0.0,
    },
    {
        "Milestone": "2. Engine & Subsystems Initialized",
        "Allocated (MB)": df_tiny.iloc[1]["Allocated (MB)"] if len(df_tiny) > 1 else 0.0,
        "Reserved (MB)": df_tiny.iloc[1]["Reserved (MB)"] if len(df_tiny) > 1 else 0.0,
        "Peak Alloc (MB)": df_tiny.iloc[1]["Peak Alloc (MB)"] if len(df_tiny) > 1 else 0.0,
    },
    {
        "Milestone": "3. ForwardRequest Staged (Tiny)",
        "Allocated (MB)": df_tiny.iloc[1]["Allocated (MB)"] if len(df_tiny) > 1 else 0.0,
        "Reserved (MB)": df_tiny.iloc[1]["Reserved (MB)"] if len(df_tiny) > 1 else 0.0,
        "Peak Alloc (MB)": df_tiny["Peak Alloc (MB)"].max() if not df_tiny.empty else 0.0,
    },
    {
        "Milestone": "4. ForwardRequest Staged (Large 64-tok)",
        "Allocated (MB)": df_large.iloc[1]["Allocated (MB)"] if len(df_large) > 1 else 0.0,
        "Reserved (MB)": df_large.iloc[1]["Reserved (MB)"] if len(df_large) > 1 else 0.0,
        "Peak Alloc (MB)": df_large["Peak Alloc (MB)"].max() if not df_large.empty else 0.0,
    },
    {
        "Milestone": "5. Post-Execution Steady State",
        "Allocated (MB)": df_large.iloc[-1]["Allocated (MB)"] if not df_large.empty else 0.0,
        "Reserved (MB)": df_large.iloc[-1]["Reserved (MB)"] if not df_large.empty else 0.0,
        "Peak Alloc (MB)": df_large.iloc[-1]["Peak Alloc (MB)"] if not df_large.empty else 0.0,
    },
]

df_summary = pd.DataFrame(summary_records)
df_summary["T4 Utilization (%)"] = ((df_summary["Reserved (MB)"] / t4_total_capacity_mb) * 100).round(2)
df_summary["Remaining Headroom (MB)"] = (t4_total_capacity_mb - df_summary["Reserved (MB)"]).round(2)

print("=" * 95)
print("MEASURED MOEINFRA ENGINE MEMORY MILESTONES")
print("=" * 95)
display(df_summary)

# 2. Projected Full INT4 Model Memory Budget on Tesla T4
print("\n" + "=" * 95)
print("PROJECTED INT4 MIXTRAL-8x7B FULL DEPLOYMENT BUDGET ON TESLA T4 (16GB)")
print("=" * 95)

proj_records = [
    {"Component": "GPU Resident Expert Slots (8 slots @ 84.0 MiB)", "Size (MB)": round(8 * 88.08, 1), "Share of T4 (%)": round((8 * 88.08 / t4_total_capacity_mb) * 100, 2)},
    {"Component": "Non-Expert Dense Layers (Attention, Embeddings, Norm)", "Size (MB)": 1800.0, "Share of T4 (%)": round((1800.0 / t4_total_capacity_mb) * 100, 2)},
    {"Component": "PyTorch CUDA Runtime / Context Baseline", "Size (MB)": 400.0, "Share of T4 (%)": round((400.0 / t4_total_capacity_mb) * 100, 2)},
    {"Component": "Estimated Static VRAM Subtotal", "Size (MB)": round(8 * 88.08 + 1800.0 + 400.0, 1), "Share of T4 (%)": round(((8 * 88.08 + 2200.0) / t4_total_capacity_mb) * 100, 2)},
    {"Component": "Dynamic Headroom for KV-Cache & Activations", "Size (MB)": round(t4_total_capacity_mb - (8 * 88.08 + 2200.0), 1), "Share of T4 (%)": round(((t4_total_capacity_mb - (8 * 88.08 + 2200.0)) / t4_total_capacity_mb) * 100, 2)},
]
df_proj = pd.DataFrame(proj_records)
display(df_proj)

headroom_mb = t4_total_capacity_mb - (8 * 88.08 + 2200.0)
print(f"\nAvailable VRAM Headroom for KV Cache & Activations: {headroom_mb:,.1f} MB ({headroom_mb / 1024:.2f} GB)")
print(f"Safety Margin on 16GB VRAM: {(headroom_mb / t4_total_capacity_mb) * 100:.1f}%")


## 14. NVIDIA Nsight Systems Profiling Guide

While `torch.cuda` and `torch.profiler` provide framework-level memory introspection, **NVIDIA Nsight Systems (`nsys`)** is the gold standard for profiling hardware PCIe saturation, CUDA stream concurrency, and CPU-to-GPU transfer bottlenecks.

---

### Why Nsight Systems for MoEInfra?
1. **PCIe Bandwidth Verification:** Measures whether asynchronous expert transfers achieve the theoretical ~8.0 GB/s on Kaggle's PCIe Gen3 x16 bus.
2. **Compute-Transfer Overlap:** Verifies whether `TransferScheduler` successfully overlaps memory transfers with layer $L$ compute while prefetching layer $L+1$.
3. **Kernel Latency:** Visualizes SwiGLU GEMM kernel launch overhead.

---

### Step 1: Instrumenting MoEInfra with NVTX Ranges
Add NVTX range markers in Python so your inference stages appear clearly labeled in the Nsight timeline:

```python
import torch.cuda.nvtx as nvtx

# In engine prefill stage
nvtx.range_push("Prefill Stage")
hidden = engine._prefill(input_ids)
nvtx.range_pop()

# In transfer scheduler
nvtx.range_push(f"PCIe_Transfer_L{layer}_E{expert}")
moved = tensor.to("cuda:0", non_blocking=True)
nvtx.range_pop()

# In engine decode step
nvtx.range_push(f"Decode_Step_{step}")
tokens = engine._decode(curr_hidden, max_new_tokens=1)
nvtx.range_pop()
```

---

### Step 2: Running `nsys` CLI on Kaggle / Linux
Run the profiling CLI directly from the terminal or a notebook cell (without needing a GUI installed on the host):

```bash
nsys profile \
    --trace=cuda,nvtx,osrt,cublas \
    --sample=none \
    --cpuctxsw=none \
    --force-overwrite=true \
    --output=moeinfra_nsys_report \
    python -c "
from config_loader import load_config
from engine.engine import InferenceEngine
from engine.types import ForwardRequest
import torch

cfg = load_config('config.yaml')
engine = InferenceEngine(cfg)
# Run inference
req = ForwardRequest(input_ids=torch.randint(0, 32000, (1, 64)), max_new_tokens=8)
try:
    engine.generate(req)
except NotImplementedError:
    pass
"
```

---

### Step 3: Analyzing the Report
1. Download the generated `moeinfra_nsys_report.nsys-rep` from Kaggle.
2. Open it in the **NVIDIA Nsight Systems GUI** on your local workstation.
3. Check the **CUDA Memory Operations** row:
   - Verify that Host-to-Device (`HtoD`) PCIe memory transfers coincide with compute kernels on separate streams.
   - Look for any GPU pipeline bubbles where compute stalls waiting for an expert weight transfer to finish.
